# EDA biblioteca



In [442]:
## exportacion de librerias

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path
import mysql.connector
from mysql.connector import Error
import requests as rq

sns.set_theme(style='whitegrid')

In [443]:
## carga de datos prestamos

df_excel_prestamos = pd.read_excel('./datos_sucios/prestamos.xlsx')
df_excel_prestamos.head()


,id_prestamo,fecha,id_libro,generos,formato,perfil_socio,sala,dias_prestamo,renovaciones,dias_retraso,comentario
0,4181,2026-04-13,L0044,novela negra;historica,Fisico,Adulto,Barrio Norte,30.0,0,15.0,NaN
1,319,2026-01-24,L0005,poesia,Fisico,Infantil,Barrio Norte,30.0,0,0.0,paginas subrayadas a lapiz
2,1316,2026-02-15,L0033,ciencia ficcion;fantasia;romantica,Fisico,Adulto,Barrio Sur,14.0,0,0.0,NaN
3,1157,2026-02-11,L0151,historica;novela negra,Ebook,Infantil,Central,7.0,0,0.0,NaN
4,34,2026-01-16,L0004,poesia;ensayo,Fisico,Senior,Barrio Sur,14.0,2,0.0,Paginas subrayadas a lapiz


In [444]:
## carga datos libros

df_excel_libros = pd.read_excel('./datos_sucios/libros.xlsx')
df_excel_libros.head()

,id_libro,titulo,autor,anio_publicacion,editorial,generos
0,L0001,El sombra del norte,Albano Llopis Hierro,1986,Anagrama,historica;biografia
1,L0002,El memoria imposible,Buenaventura de Bonet,1959,Alfaguara,fantasia
2,L0003,El isla de arena,Ileana AntÃ³n-AndrÃ©s,1958,Salamandra,biografia;poesia
3,L0004,El codigo de arena,CÃ©sar Guerrero Vazquez,1990,Alfaguara,poesia;ensayo
4,L0005,La viaje de medianoche,Alba Mar Flor Rivas,1998,Alfaguara,poesia


In [445]:
## primera exploracion prestamos

print('dimensiones en filas y columnas', df_excel_prestamos.shape) # da informacion del numero de filas y columnas
df_excel_prestamos.info() # resumen de la estructura de un DataFrame.

## resumen estadistico prestamos

df_excel_prestamos.describe().round(2) # resumen estadistico de el dataframe

dimensiones en filas y columnas (10075, 11)
<class 'pandas.DataFrame'>
RangeIndex: 10075 entries, 0 to 10074
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id_prestamo    10075 non-null  int64         
 1   fecha          10075 non-null  datetime64[us]
 2   id_libro       10075 non-null  str           
 3   generos        10075 non-null  str           
 4   formato        10075 non-null  str           
 5   perfil_socio   9778 non-null   str           
 6   sala           10075 non-null  str           
 7   dias_prestamo  9775 non-null   float64       
 8   renovaciones   10075 non-null  int64         
 9   dias_retraso   9746 non-null   str           
 10  comentario     1463 non-null   str           
dtypes: datetime64[us](1), float64(1), int64(2), str(7)
memory usage: 865.9 KB


,id_prestamo,fecha,dias_prestamo,renovaciones
count,10075.00,10075,9775.00,10075.00
mean,4998.84,2026-04-23 11:18:11.612903,17.24,0.46
min,1.00,2026-01-16 00:00:00,7.00,0.00
25%,2502.50,2026-03-12 00:00:00,14.00,0.00
50%,4996.00,2026-04-27 00:00:00,14.00,0.00
75%,7498.50,2026-06-07 00:00:00,21.00,1.00
max,10000.00,2026-07-14 00:00:00,30.00,2.00
std,2886.94,NaN,7.82,0.67


In [446]:
## primera exploracion

print('dimensiones en filas y columnas', df_excel_libros.shape) # da informacion del numero de filas y columnas
df_excel_libros.info() # resumen de la estructura de un DataFrame.

# resumen estadistico libros

df_excel_libros.describe().round(2) # resumen estadistico de el dataframe

dimensiones en filas y columnas (200, 6)
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id_libro          200 non-null    str  
 1   titulo            200 non-null    str  
 2   autor             200 non-null    str  
 3   anio_publicacion  200 non-null    int64
 4   editorial         200 non-null    str  
 5   generos           200 non-null    str  
dtypes: int64(1), str(5)
memory usage: 9.5 KB


,anio_publicacion
count,200.00
mean,1991.09
std,20.72
min,1955.00
25%,1973.00
50%,1993.50
75%,2009.00
max,2025.00


In [447]:
## nulos por columna

print(f'Hay {df_excel_prestamos.isnull().sum()} nulos en el archivo de prestamos') # Calcula y muestra el total de valores nulos o faltantes por cada columna dentro de df_excel_prestamos.

## filas duplicada

print('filas duplicadas', df_excel_prestamos.duplicated().sum())

Hay id_prestamo         0
fecha               0
id_libro            0
generos             0
formato             0
perfil_socio      297
sala                0
dias_prestamo     300
renovaciones        0
dias_retraso      329
comentario       8612
dtype: int64 nulos en el archivo de prestamos
filas duplicadas 75


In [448]:
## nulos por columna libros

print(f'Hay {df_excel_libros.isnull().sum()} nulos en el archivo de libros') # Muestra el total de nulos por columna dentro de df_excel_libros.

## filas duplicadas:

print('filas duplicadas', df_excel_libros.duplicated().sum())

Hay id_libro            0
titulo              0
autor               0
anio_publicacion    0
editorial           0
generos             0
dtype: int64 nulos en el archivo de libros
filas duplicadas 0


In [449]:
## limpieza

from lib.limpieza import (
    limpieza_generos,
    limpieza_dias_retraso,
    comentario,
    limpieza_dias_prestamo,
    limpieza_perfil_socio,
    normalizacion_id_prestamo,
    limpieza_generos_l,
    id_libro,
    limpieza_autor,
    estadisticas_descriptivas,
    detectar_outliers,
    generar_columnas_derivadas
)

In [450]:
## creacion archivos limpios

output_dir = Path("datos_limpios")
output_dir.mkdir(exist_ok=True) # crea la carpeta si no existe

# Limpiar libros
libros_limpios = df_excel_libros.copy() # Se hace una copia del DataFrame para no modificar el original.
libros_limpios = limpieza_generos_l(libros_limpios) 
libros_limpios = id_libro(libros_limpios)
libros_limpios = limpieza_autor(libros_limpios)

libros_limpios.to_excel(
    output_dir / "libros_limpios.xlsx",
    index=False, # Hace que no se escriba el índice de pandas en el Excel. 
    engine="openpyxl" # indica que se usa esa biblioteca para crear excel
)

prestamos_limpios = df_excel_prestamos.copy()
prestamos_limpios = limpieza_generos(prestamos_limpios)
prestamos_limpios = limpieza_dias_prestamo(prestamos_limpios)
prestamos_limpios = limpieza_dias_retraso(prestamos_limpios)
prestamos_limpios = comentario(prestamos_limpios)
prestamos_limpios = limpieza_perfil_socio(prestamos_limpios)
prestamos_limpios = normalizacion_id_prestamo(prestamos_limpios)

# Estadísticas, outliers y columnas derivadas
prestamos_limpios = estadisticas_descriptivas(prestamos_limpios)
prestamos_limpios = detectar_outliers(prestamos_limpios)
prestamos_limpios = generar_columnas_derivadas(prestamos_limpios)

prestamos_limpios.to_excel(output_dir / "prestamos_limpios.xlsx", index=False, engine='openpyxl')


ESTADÍSTICAS DESCRIPTIVAS

ID_PRESTAMO:
  Media: 5000.50
  Mediana: 5000.50
  Desviación estándar: 2886.75
  Mínimo: 1.00
  Máximo: 10000.00
  Q1 (25%): 2500.75
  Q3 (75%): 7500.25

DIAS_PRESTAMO:
  Media: 17.23
  Mediana: 14.00
  Desviación estándar: 7.70
  Mínimo: 7.00
  Máximo: 30.00
  Q1 (25%): 14.00
  Q3 (75%): 21.00

RENOVACIONES:
  Media: 0.46
  Mediana: 0.00
  Desviación estándar: 0.67
  Mínimo: 0.00
  Máximo: 2.00
  Q1 (25%): 0.00
  Q3 (75%): 1.00

DIAS_RETRASO:
  Media: 2.22
  Mediana: 0.00
  Desviación estándar: 10.85
  Mínimo: 0.00
  Máximo: 700.00
  Q1 (25%): 0.00
  Q3 (75%): 0.00

DETECCIÓN DE OUTLIERS (IQR)

ID_PRESTAMO: Sin outliers detectados

DIAS_PRESTAMO: Sin outliers detectados

RENOVACIONES: Sin outliers detectados

DIAS_RETRASO: 2264 outliers detectados
  Rango válido: [0.00, 0.00]
  Reemplazados por la mediana: 0.00

GENERANDO COLUMNAS DERIVADAS

✓ Columna 'categoria_duracion' creada
✓ Columnas 'tiene_retraso' y 'categoria_retraso' creadas
✓ Columna 'perfil_rie

In [451]:
## conexion  a base de datos y creacion de resumenes y columnas agregadas

from lib.conexion_BBDD import (
    get_connection,
    insert_prestamos, 
    insert_libros, 
    validar_carga_prestamos, 
    validar_carga_libros, 
    crear_resumen_prestamos, 
    crear_resumen_generos
)


In [452]:
## graficos interactivos

### evolucion temporal de los prestamos

def obtener_datos_tiempo():
    conn = get_connection()
    query = "select fecha, count(*) as cantidad_prestamos from prestamos group by fecha order by fecha"
    df = pd.read_sql(query, conn)
    conn.close()
    return df

df = obtener_datos_tiempo()

plt.figure(figsize=(10,12))
fig = px.line(
    df,
    x="fecha",
    y="cantidad_prestamos",
    title="Evolución temporal de préstamos",
    markers=True, # añadir puntos en la uniones
)
fig.show()

fig.write_html('line_evolucion_temporal_prestamos.html')



C:\Users\UsuarioM\AppData\Local\Temp\ipykernel_9324\2301878496.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


<Figure size 1000x1200 with 0 Axes>

In [453]:
### grafico de comparacion (prestamos totales por formato)

def obtener_comparacion():
    conn=get_connection()
    query = "select formato, count(id_prestamo) as total_prestamos from prestamos group by formato"
    df = pd.read_sql(query, conn)
    conn.close()
    return df

df = obtener_comparacion()

plt.figure(figsize=(12,6))
fig = px.bar(
    df, 
    x='formato', 
    y='total_prestamos', 
    title='Comparativa de Préstamos por formato',
    color='formato',
    text_auto=True
)

fig.show()

fig.write_html('barras_comparacion_prestamos_formato.html')

C:\Users\UsuarioM\AppData\Local\Temp\ipykernel_9324\3077065336.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


<Figure size 1200x600 with 0 Axes>

In [454]:
### grafico de distribucion de los prestamos por perfil

def obtener_distribucion():
    conn = get_connection()
    query = """
        select perfil_socio, count(*) as cantidad
        from prestamos
        group by perfil_socio
        order by perfil_socio
    """
    df = pd.read_sql(query, conn)
    conn.close()
    return df


df_3 = obtener_distribucion()

fig = px.pie(
    data_frame=df_3,
    names='perfil_socio',
    values='cantidad',
    title='Préstamos por perfil socio'
)

fig.show()

fig.write_html("pie_distribucion_prestamos_perfil.html")

C:\Users\UsuarioM\AppData\Local\Temp\ipykernel_9324\3203117909.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [455]:
## merge

# 1. Carga de los archivos limpios
df_prestamos_limpio = pd.read_excel('./datos_limpios/prestamos_limpios.xlsx')
df_libros_limpios = pd.read_excel('./datos_limpios/libros_limpios.xlsx')

# 2. Unión (merge) de ambas tablas por 'id_libro'
df = pd.merge(
    df_prestamos_limpio,
    df_libros_limpios,
    on='id_libro',
    how='left'
).rename(columns={'generos_x': 'generos'})


In [456]:
# informe automatizado

# 3. Conversión de los datos unificados a JSON
json_data = df.to_json(
    orient="records",
    date_format="iso"
)

# Top 10 libros más prestados
top10_libros = (
    df.groupby(["id_libro", "titulo", "autor"])
      .size()
      .reset_index(name="num_prestamos")
      .sort_values("num_prestamos", ascending=False)
      .head(10)
)

libros_sin_prestamos = (
    df_libros_limpios[
        ~df_libros_limpios["id_libro"].isin(df_prestamos_limpio["id_libro"])
    ][["titulo", "autor", "generos", "editorial"]]
)

# Préstamos por género
prestamos_genero = (
    df.groupby("generos")
      .size()
      .reset_index(name="num_prestamos")
      .sort_values("num_prestamos", ascending=False)
)

prestamos_genero["porcentaje"] = (
    prestamos_genero["num_prestamos"] /
    prestamos_genero["num_prestamos"].sum() * 100
).round(2)

# Convertir préstamos por género a texto con guiones
prestamos_genero_texto = ""

for _, fila in prestamos_genero.head(10).iterrows():
    prestamos_genero_texto += (
        f"- {fila['generos']}: "
        f"{fila['num_prestamos']} préstamos "
        f"({fila['porcentaje']}%)\n"
    )

# Autores más prestados
autores_top = (
    df.groupby("autor")
      .size()
      .reset_index(name="num_prestamos")
      .sort_values("num_prestamos", ascending=False)
      .head(10)
)

# Editoriales más prestadas
editoriales_top = (
    df.groupby("editorial")
      .size()
      .reset_index(name="num_prestamos")
      .sort_values("num_prestamos", ascending=False)
      .head(10)
)

# Convertir editoriales más prestadas a texto con guiones
editoriales_top_texto = ""

for _, fila in editoriales_top.head(6).iterrows():
    editoriales_top_texto += (
        f"- {fila['editorial']}: "
        f"{fila['num_prestamos']} préstamos\n"
    )

# 4. Diccionario de resumen automatizado (con el mismo formato exacto)
resumen = {

    "num_libros": len(df_libros_limpios),

    "num_prestamos": len(df_prestamos_limpio),

    "top10_libros": top10_libros.to_dict(orient="records"),

    "libros_sin_prestamos": libros_sin_prestamos.to_dict(orient="records"),

    "prestamos_genero": prestamos_genero_texto,

    "autores_top": autores_top.to_dict(orient="records"),

    "editoriales_top": editoriales_top_texto
}

URL_informe = "https://miriam.n8ncamp.com/webhook-test/montar_informe"

response = rq.post(URL_informe, json=resumen)

conclusiones = response.json()

fichero = open("./conclusiones/conclusiones.txt", 'w', encoding='UTF-8')
fichero.write(conclusiones['mensaje'])
fichero.close()